## Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 -- [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Python for Data Analysis*, Chapter 10 - Data Aggregation and Group Operations. The notes below follow the book fairly closely, so keep a notebook open and try the commands as you go.

## Career Reading (discuss Thursday 10/1)

*Build a Career in Data Science*, 2.1 Data Science Companies: Massive Tech. Read it, take notes, and come to class on Thursday ready to talk about it.

## Data Aggregation and Groups

Applying functions to separate groups of your data can be a critical component of data analysis. Often we have questions about how subgroups of the data differ or we might want to compute pivot tables for reporting or visualization. We are going to get more in depth into the `groupby()` function and really see if we can understand what it is doing and what object is returned from it. We will also see how to compute pivot tables and cross-tabulations. 

In [ ]:
# Some basic package imports
import os
import numpy as np
import pandas as pd

# Visualization packages
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
import seaborn as sns

## Grouping

When grouping we use *split-apply-combine* to describe group operations. 

- SPLIT - we first split the data based on one or more keys. Think about categorical values in a single column.
- APPLY - now we apply a function to each of the data subsets that we split above.
- COMBINE - finally the results of these functions are combined into a single summary output or result object.

Here is some example data

In [ ]:
df = pd.DataFrame({"key1" : ["a", "a", None, "b", "b", "a", None],
                   "key2" : pd.Series([1, 2, 1, 2, 1, None, 1],
                                      dtype="Int64"),
                   "data1" : np.random.standard_normal(7),
                   "data2" : np.random.standard_normal(7)})
df

What if you wanted to calculate the mean of the column data1, but on subsets or groups based on key1.

In [ ]:
# First group the data
grouped = df["data1"].groupby(df["key1"])

# What is this object
grouped

Notice that we can't see our data here. This has created a python object that contains all the information we need to apply a function, but it has not actually computed anything yet. We have prepared our split.

Now we will apply a function!

In [ ]:
grouped.sum()

We have now applied a function and combined the results into a series.

### More Examples

In [ ]:
# Calculate the mean of the data in column data1
# Group by two keys
# Results in hierarchical index
df["data1"].groupby([df["key1"], df["key2"]]).mean()

In [ ]:
# Calculate the means of both column data1 and data2
# Group by both key1 and key2
df.groupby([df["key1"], df["key2"]]).mean()

In [ ]:
# Group by key1 and then ask the size of each group
# This will automatically drop NaNs
df.groupby('key1').size()

In [ ]:
df.groupby('key1',dropna=False).size()

In [ ]:
# Count the number of nonnull values
df.groupby('key1').count()

### You Try

Try applying some other functions that are available to the groupby object. See if you can figure out what the function is doing. To see all the possible functions type `grouped.` and then hit tab!

In [ ]:
# Your code here


### Iterating over Groups

You can iterate over the group object returned from groupby().

In [ ]:
grouped = df["data1"].groupby(df["key1"])

for g in grouped:
    print(g[0])
    display(g[1])

You can see that each thing in our grouped object is a tuple. The first entry in the tuple is the group name (or category) the second object is a Series or a DataFrame depending on the number of columns sent in.

In [ ]:
grouped = df[["data1","data2"]].groupby(df["key1"])

for g in grouped:
    print(g[0])
    display(g[1])

In [ ]:
# If you group by two keys, your group names have two values.
grouped = df[["data1","data2"]].groupby([df["key1"],df["key2"]])

for g in grouped:
    print(g[0])
    display(g[1])

### Column vs Row Grouping

By default groups are created on the axis='index' meaning that it is breaking up the rows into groups based on labels in a column. But we could break up columns based on values in a row. 

Here is an example where we group the data based on column names.

1. We take the transpose of the dataframe.
2. Here we send in a dictionary that maps the values found in the index to either key or data.
3. The groupby now splits based on our old column names.

In [ ]:
df.T

In [ ]:
# Here we map the names found in the index
# To new names - we will group on the new names.
mapping = {"key1": "key", "key2": "key",
            "data1": "data", "data2": "data"}

grouped = df.T.groupby(mapping)

for g in grouped:
    print(g[0])
    display(g[1])

### Grouping with Functions

You can also use python functions to specify groups. 

For example, in the data below we have information about different people. What if you wanted to group a data set based on the length of their name. You can do that!

In [ ]:
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people

In [ ]:
grouped = people.groupby(len)

for g in grouped:
    print(g[0])
    display(g[1])

### You Try

Read in the macro data - the same data we used on Day 6 - just run the code below.

Then group the data by year and apply a function that makes sense with this data. Something like mean, max, min, etc...

Finally plot the resulting time-series data. For example, the average cpi over time.

In [ ]:
macro = pd.read_csv("data/macrodata.csv")
data = macro[["year","quarter","cpi", "m1", "tbilrate", "unemp"]]

In [ ]:
# Your code here


## Data Aggregation

Data aggregation is when you take an array (or list) of values and apply a transformation that produces a single scalar output. Think about a list of grades and then taking an average.


**Functions that are optimized for the groupby() method**

| Aggregation Function | Description                                                                 | Example Usage                   |
|-----------------------|-----------------------------------------------------------------------------|---------------------------------|
| `sum`                | Sum of values (ignores NaN)                                                 | `df.groupby("col")["val"].sum()` |
| `mean`               | Mean (average) of values (ignores NaN)                                      | `df.groupby("col")["val"].mean()` |
| `median`             | Median (50th percentile)                                                    | `df.groupby("col")["val"].median()` |
| `min`                | Minimum value                                                               | `df.groupby("col")["val"].min()` |
| `max`                | Maximum value                                                               | `df.groupby("col")["val"].max()` |
| `count`              | Count of non-NaN values                                                     | `df.groupby("col")["val"].count()` |
| `size`               | Count of all rows (including NaN)                                           | `df.groupby("col").size()`        |
| `std`                | Standard deviation (ddof=1 by default)                                      | `df.groupby("col")["val"].std()` |
| `var`                | Variance (ddof=1 by default)                                                | `df.groupby("col")["val"].var()` |
| `prod`               | Product of values                                                           | `df.groupby("col")["val"].prod()` |
| `first`              | First non-NaN value in group                                                | `df.groupby("col")["val"].first()` |
| `last`               | Last non-NaN value in group                                                 | `df.groupby("col")["val"].last()` |
| `nunique`            | Number of distinct values    | `df.groupby("col")["val"].nunique()` |

**Functions that can be applied to groupby but are a bit slower**

| Aggregation Function | Description                                  | Example Usage                        | Notes                                   |
|-----------------------|----------------------------------------------|--------------------------------------|-----------------------------------------|
| `mode`               | Most frequent value(s) in group              | `df.groupby("col")["val"].agg(pd.Series.mode)` | Can return multiple values per group |
| `skew`               | Sample skewness of distribution              | `df.groupby("col")["val"].skew()`     | Slower, especially on large groups     |
| `kurt` / `kurtosis`  | Kurtosis (tailedness of distribution)        | `df.groupby("col")["val"].kurt()`     | Not Cython-optimized                   |
| `quantile`           | Value at given quantile (e.g. 0.25, 0.75)    | `df.groupby("col")["val"].quantile(0.25)` | Flexible but slower                   |
| `sem`                | Standard error of the mean                   | `df.groupby("col")["val"].sem()`      | Based on `std / sqrt(n)`               |
| `describe`           | Multiple summary stats at once               | `df.groupby("col")["val"].describe()` | Returns count, mean, std, min, etc.    |
| Custom functions     | Any user-defined aggregation (via `.apply`)  | `df.groupby("col")["val"].apply(func)`| Flexible but usually much slower        |


Now sometimes you want to apply multiple functions to a single grouped object. One way to do this is with the `.agg()` function. Let's read in the tips data we have seen before:

In [ ]:
tips = pd.read_csv("data/tips.csv")
tips.head()

Now lets say that we want to understand the tip percentage from different groups: smokers vs nonsmokers, vs day of the week. Maybe you want to know the average and standard deviation of the tip percentage and you want to define a calculation of your own called max_to_min that calculates the difference between the max and min percent.

In [ ]:
# First we have to add the percentage to the data
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

In [ ]:
# Now we can define our own custom function
def max_to_min(arr):
    return arr.max() - arr.min()

In [ ]:
# Next lets group the data
grouped = tips.groupby(["day", "smoker"])

In [ ]:
# And finally apply the functions using the .agg() 
functions = ["mean", "std", max_to_min]
agg_data = grouped['tip_pct'].agg(functions)
agg_data

#### This is a case where I would use pandas .plot()

In [ ]:
# Graph the results
agg_data.plot.bar()
plt.grid()
plt.show()

In [ ]:
# You can also return the data without the groups as and index
# just use the as_index=False command
grouped2 = tips.groupby(["day", "smoker"],as_index=False)
functions = ["mean", "std", max_to_min]
agg_data2 = grouped2['tip_pct'].agg(functions)
agg_data2

From this example hopefully you can see how powerful grouping and aggregating can be in quickly comparing parts of your dataset!

### You Try

What happens when you run the following command using the tip data above? Can you predict what the output will be before you run the code? Then run the code and explain the results. Finally make a plot - your choice on the type.

    grouped = tips.groupby(["day", "smoker","time"])
    functions = ["count", "mean", "max"]
    result = grouped[["tip_pct", "total_bill"]].agg(functions)
    result

In [ ]:
# Your code here

## Quantile and Bucket Analysis

We have seen some functions that can help us to cut our data into buckets, `pd.cut()` for example. Another similar function is `pd.qcut()` which divides your data into sample quantiles. You can use these functions on grouped data.

Below we will use `.qcut()` to get quartile categories and then group by the quartiles. Finally we can aggregate over that data.

In [ ]:
quartiles = pd.qcut(tips['total_bill'],4)
tips['quartiles'] = quartiles
tips.head()

In [ ]:
grouped = tips['total_bill'].groupby(tips['quartiles'])
grouped.agg(["count","max","min","mean"])

## More Examples of using groupby()

### Filling Missing Values

In [ ]:
states = ["Ohio", "New York", "Vermont", "Florida",
          "Oregon", "Nevada", "California", "Idaho"]
location = ["East", "East", "East", "East",
             "West", "West", "West", "West"]
data = pd.DataFrame({'Data':np.random.standard_normal(8),
                     'Location':location}, 
                    index=states)

# Add some NaNs
for s in ["Vermont", "Nevada", "Idaho"]:
    data.loc[s, "Data"] = np.nan
data

What if we wanted to fill these NaNs, but with different values based on whether they are in the east or west. Maybe we want to average the east and west values and use that number to fill the NaNs.

In [ ]:
# Define a function that will return the fill value
def fill_mean(group):
    return group.fillna(group.mean())

# We group by location
grouped = data.groupby('Location')
# Then fill in the NANs with the averages of the group
grouped.apply(fill_mean)

### Groupwise Linear Regression

We talked about linear regression in DATA101 and DATA100. It is a way to fit a straight line to some given data.

We will use sklearn to do this. If you don't have it installed, you should install it.

In [ ]:
# !conda install -y scikit-learn

In [ ]:
# Get some data
close_px = pd.read_csv("data/stock_px.csv", parse_dates=True,
                       index_col=0)

# The .info() command gives you information about the data
# Including data types and number of nonnulls
close_px.info()

In [ ]:
close_px.head()

In [ ]:
# Lets group the data by year
# First confirm that the index values are dates
first_index = close_px.index[0]
print(first_index)
print(type(first_index))

In [ ]:
# This is a timestamp object
# You can do first_index. and press tab to see all the options
print(first_index.year)
print(first_index.month)
print(first_index.month_name())
print(first_index.day)
print(first_index.day_name())

In [ ]:
# So first write a function that will return the year
def get_year(x):
    return x.year

# Then group by the year
by_year = close_px.groupby(get_year)

In [ ]:
from sklearn.linear_model import LinearRegression
# Define a linear regression and return intercept and slope
def regress(data,xvars,yvar):
    X = data[xvars]
    y = data[yvar]
    LM = LinearRegression()
    LM.fit(X, y)
    slope = LM.coef_[0]
    intercept = LM.intercept_
    return intercept, slope

# Apply the linear regression to the groups
by_year.apply(regress,yvar='AAPL',xvars=['SPX'])

## Pivot Tables and Cross-Tabulation

### Pivot Table

Pivot tables are often found in spreadsheet programs and are a way to summarize data. We have seen the `.pivot` operation as a way to wrangle the data. Here we will look at the `pivot_table()` method. The results in many cases can be produced using the groupby function, but this acts as a shortcut and can add partial totals or margins to the data.

In [ ]:
# Lets use the same tips data that we loaded in above
tips = pd.read_csv("data/tips.csv")
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.head()

In [ ]:
tips.pivot_table(index=["day", "smoker"],
                 values=["size", "tip", "tip_pct", "total_bill"])

**NOTE** by default the pivot table returns the mean()

We could have done the same operation with groupby!

Arguments you might want to pass into the pivot_table() function:

- index -- the values that you are grouping by
- values -- the numbers you are aggregating
- columns -- categories to subset the columns - adding extra columns to the output
- margins=True -- include the margin or the value for the whole
- aggfunc -- aggregation function if you want something other than mean.
- fill_value -- what you want to fill in if the computation runs into a NaN

Here are some examples:

In [ ]:
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"], margins=True)

In [ ]:
tips.pivot_table(index=["time", "day"], columns="smoker",
                 values=["tip_pct", "size"], aggfunc="max")

In [ ]:
tips.pivot_table(index=["time", "size", "smoker"], columns="day",
                 values="tip_pct")

In [ ]:
tips.pivot_table(index=["time", "size", "smoker"], columns="day",
                 values="tip_pct",fill_value=0)

### Crosstab

Cross tabulation is a type of pivot table that returns frequency observations. You can very quickly reach into your data and get counts of the number of observations that fall into each subset.


In [ ]:
cdata = pd.crosstab([tips["time"], tips["day"]], tips["smoker"])
cdata

In [ ]:
cdata.plot.bar()
plt.grid()
plt.ylabel('Customer Count')
plt.show()

## Homework 8

The full assignment is in `HW_day8.ipynb` in your sandbox. The short version: the Kaggle video game sales data, a discussion of where the data comes from, and then an analysis using the grouping tools from today (crosstab, groupby with aggregation, and plots of the results).

Work the problems in your sandbox. Your team's write-up notebook goes in `Week05` of your team repo. Homework 8 and Homework 9 both go in `Week05` and are due Sunday 10/4 at 11:59pm.